# Creating a Simple Agent with Tracing

In [ ]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

In [3]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [5]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give right answers and concise answers.
    """,
)

Let's execute the Agent:

In [6]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Bananas are a healthy, convenient fruit. Key points:
    
    - Nutrients: good source of potassium, vitamin C, vitamin B6, and dietary fiber; low in fat.
    - Energy: provide quick energy from natural sugars and carbohydrates; ripeness affects sugar level.
    - Health perks: support heart health, digestion, and satiety; fiber helps with regularity.
    - Considerations: portion size if watching sugar or carbohydrate intake; diabetics should pair with protein/fat and monitor portions.
    - Others: some people with latex allergies may react to bananas.
    
    Average medium banana (~118 g) has about 105 calories. Want a specific daily amount or dietary context?
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [8]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Bananas are a healthy, convenient fruit. Key points:

- Nutrients: potassium, vitamin B6, vitamin C, fiber.
- Health benefits: supports heart health, aids digestion, provides steady energy.
- Carbs/sugars: natural sugars; about 14 g per medium banana.
- Variants: unripe (more resistant starch) vs ripe (more simple sugars).
- Considerations: okay for most; watch portions if you have kidney disease or diabetes; allergy possible.

Overall: a good, nutrient-dense snack when eaten as part of a balanced diet.

_Good Job!_